<a href="https://colab.research.google.com/github/amit-sahu-a11y/ML_projects_for_practice/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Identity — MasterCustomerID, CustomerID

Why: This is your grouping key — nothing to "convert," but note CustomerID is the sub-entity and MasterCustomerID is the rollup parent (one master can have multiple CustomerIDs, e.g., subsidiaries). Everything below groups by MasterCustomerID.

In [ ]:
df.groupby("MasterCustomerID")

2. Segment Label — MC_Strategy_Segment


Why: This is your target/label, not a feature. Just take it once per MasterCustomerID (it shouldn't vary within a customer, but check):

In [ ]:
segment = df.groupby("MasterCustomerID")["MC_Strategy_Segment"].first()
# sanity check it's constant:
assert df.groupby("MasterCustomerID")["MC_Strategy_Segment"].nunique().max() == 1

3. Geography — RL_Market, RL_Office, Client_Market, Client_Office

Why: A customer can have multiple rows with different offices (e.g., serviced by different regional teams for different products), so you need (a) the dominant/most common assignment, and (b) a measure of how well internal servicing matches the client's real location.

Dominant market/office (mode):

In [ ]:
dominant_rl_market = df.groupby("MasterCustomerID")["RL_Market"].agg(lambda x: x.mode()[0])
dominant_client_market = df.groupby("MasterCustomerID")["Client_Market"].agg(lambda x: x.mode()[0])

Formula reason: mode = most frequent category. Mean/median don't apply to text categories, so mode is the standard way to get a single representative value from repeated categorical rows.

Market mismatch (row-level flag, then rolled up):

Formula reason: This is a rate, not a count, because customers have different numbers of transactions — a customer with 100 rows and 20 mismatches (20%) is very different from one with 5 rows and 2 mismatches (40%), even though raw counts might look similar. Mean of a 0/1 flag = proportion.


In [ ]:
df["market_mismatch"] = (df["RL_Market"] != df["Client_Market"]).astype(int)

market_mismatch_rate = df.groupby("MasterCustomerID")["market_mismatch"].mean()
# = (number of mismatched rows) / (total rows) for that customer

4. Firm Classification — Industry, Sector, Segments, Strata, Employee_range


Why: Same logic as geography — these describe the client's own business, and should be constant per client, but data mess (from the decks: nulls, "Not Known" categories) means you need both a dominant value and a diversity count.

Dominant category:

In [ ]:
dominant_industry = df.groupby("MasterCustomerID")["Industry"].agg(lambda x: x.mode()[0])
dominant_sector = df.groupby("MasterCustomerID")["Sector"].agg(lambda x: x.mode()[0])

Diversity count (this is literally H2's "relationship complexity" feature):

In [ ]:
num_unique_industries = df.groupby("MasterCustomerID")["Industry"].nunique()
num_unique_sectors = df.groupby("MasterCustomerID")["Sector"].nunique()

Formula reason: nunique() counts distinct categories — this measures breadth, which is a totally different signal from "which one is dominant." The decks found Gold clients average 3.24 industries vs. Platinum's 2.50 — that number comes directly from this formula.

Employee_range / Strata / Segments: these are firmographic size buckets — typically static per client, so just take mode, but flag if it changes over years (which itself could be a feature — "did this client grow into a bigger size bracket?"):

In [ ]:
size_changed_flag = df.groupby("MasterCustomerID")["Employee_range"].nunique() > 1

5. Product/Service Taken — ProductLineType, productlineid, ProductLine, Managed_Services, LOB, ServiceLine, Capability, SolutionSet, Subfunction

Why: This is the heart of H1 (product adoption breadth). Each of these is a category of "what did they buy," at increasing levels of granularity (LOB is broadest, Subfunction is most granular). The transformation is distinct count, exactly like industry diversity, but first you must apply the business rule from the project: exclude rows where Net_Services < $1,000 (treated as non-purchases/adjustments).

In [ ]:
# Step 1: apply business rule filter
df_valid = df[(df["Net_Services"] >= 1000) & (df["ProductLine"].notna())]

# Step 2: distinct counts per customer
distinct_products = df_valid.groupby("MasterCustomerID")["ProductLine"].nunique()
distinct_subfunctions = df_valid.groupby("MasterCustomerID")["Subfunction"].nunique()
distinct_solutionsets = df_valid.groupby("MasterCustomerID")["SolutionSet"].nunique()
distinct_lob = df_valid.groupby("MasterCustomerID")["LOB"].nunique()
distinct_servicelines = df_valid.groupby("MasterCustomerID")["ServiceLine"].nunique()
distinct_capabilities = df_valid.groupby("MasterCustomerID")["Capability"].nunique()

Formula reason: nunique() again — you want "how many different things," not "how many times." A customer who bought the same product 50 times isn't more diversified than one who bought 5 different products once each.

Managed Services flag:

Formula reason: .any() = boolean OR across rows — you just need to know if it ever happened, not how often

In [ ]:
has_managed_services = df.groupby("MasterCustomerID")["Managed_Services"].apply(lambda x: (x == "Yes").any())

6. Financials/Time — PeriodFiscalYear, Net_Services

Why: This is the most important group per the feature-importance finding in the decks — revenue intensity is the #1 predictor. You need total, average, tenure, and trend, not just a single sum.

In [ ]:
grp = df.groupby("MasterCustomerID")

total_net_services = grp["Net_Services"].sum()
avg_net_services = grp["Net_Services"].mean()
years_active = grp["PeriodFiscalYear"].nunique()
min_year = grp["PeriodFiscalYear"].min()
max_year = grp["PeriodFiscalYear"].max()

net_services_per_year = total_net_services / years_active

Growth trend (slope of Net_Services over years) — use linear regression per customer:

In [ ]:
from scipy.stats import linregress

def trend_slope(sub_df):
    yearly = sub_df.groupby("PeriodFiscalYear")["Net_Services"].sum().reset_index()
    if len(yearly) < 2:
        return 0  # not enough years to compute a trend
    slope, intercept, r, p, se = linregress(yearly["PeriodFiscalYear"], yearly["Net_Services"])
    return slope

net_services_trend = df.groupby("MasterCustomerID").apply(trend_slope)

Formula reason: Sum/average tell you how much, but slope tells you direction — a client at $500K flat for 3 years is different from one that grew from $200K to $800K, even though totals could be similar. This is exactly what H4 (revenue trajectory) tests.

Handling nulls in Net_Services (important — do NOT fill with median):

In [ ]:
# Per the project's own conclusion: nulls in Net_Services on rows with real product activity
# mean "no revenue recorded that year", not missing data. Keep as NaN and exclude from sum/mean
# (pandas sum/mean already skip NaN by default), but flag it separately:
null_net_services_rows = df.groupby("MasterCustomerID")["Net_Services"].apply(lambda x: x.isna().sum())

7. Flags — is_mc, is_private

Why: These are already boolean per row but might vary across a customer's rows (e.g., a master customer might have both private and non-private sub-entities). Use .any() or .mode() depending on what makes business sense:

In [ ]:
is_private_customer = df.groupby("MasterCustomerID")["is_private"].agg(lambda x: x.mode()[0])
is_mc_customer = df.groupby("MasterCustomerID")["is_mc"].agg(lambda x: x.mode()[0])

Putting It All Together

Once every group is aggregated, you join/merge them all on MasterCustomerID into one wide table — one row per customer, one column per engineered feature, plus the MC_Strategy_Segment label. That final table is what you'd run H1–H5 statistical tests on, or feed into any model.

In [ ]:
final = segment.to_frame("segment") \
    .join(dominant_rl_market) \
    .join(market_mismatch_rate) \
    .join(dominant_industry) \
    .join(num_unique_industries) \
    .join(distinct_products) \
    .join(total_net_services) \
    .join(net_services_trend) \
    .join(is_private_customer)
    # ... etc for every feature above